# DNA-Based Semantic Search: Differentiable Biophysical Representation Learning
**Project:** ChemiSearch — encoding natural language semantics into physical DNA sequences via constrained thermodynamic optimisation.
**Target:** Q1 Journal (Nature Methods / Bioinformatics / IEEE TCBB)

## Cell 1 — Reproducibility & Environment Setup


In [ ]:
import os, gc, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.random_projection import GaussianRandomProjection
from typing import Tuple, Dict, Any, Optional
import warnings
warnings.filterwarnings('ignore')

# ── 1. Seed locking ───────────────────────────────────────────────────
SEED = 42
def seed_everything(seed: int) -> None:
    """Locks every random source for full reproducibility across runs."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)          # FIX: covers multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# ── 2. Workspace ──────────────────────────────────────────────────────
WORKSPACE = './dna_search_workspace'
for sub in ('data', 'models', 'figures'):
    os.makedirs(os.path.join(WORKSPACE, sub), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}  |  Seed: {SEED}  |  PyTorch: {torch.__version__}")


## Cell 2 — Data Acquisition


In [ ]:
import pandas as pd

def load_datasets() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Downloads standard NLP benchmarks from HuggingFace Hub.
    No manual data manipulation — all data is verifiable and public.
    """
    print('Loading datasets...')
    stsb     = load_dataset('mteb/stsbenchmark-sts')
    train_df = pd.DataFrame(stsb['train'])
    val_df   = pd.DataFrame(stsb['validation'])

    nli_raw  = load_dataset('sentence-transformers/all-nli', 'triplet', split='train')
    nli_df   = pd.DataFrame(nli_raw.shuffle(seed=SEED).select(range(30000)))

    biosses_df = pd.DataFrame(load_dataset('mteb/biosses-sts', split='test'))
    stsb_test  = pd.DataFrame(stsb['test'])

    print(f"STS-B train: {len(train_df)}  |  AllNLI: {len(nli_df)}")
    print(f"STS-B test:  {len(stsb_test)}  |  BIOSSES: {len(biosses_df)}")

    # ── FIX 1: normalise STS-B scores 0-5 → 0-1 so all targets share same scale
    for df in (train_df, val_df, stsb_test):
        df['score'] = df['score'] / 5.0

    return train_df, val_df, nli_df, biosses_df, stsb_test

train_df, val_df, nli_df, biosses_df, stsb_test = load_datasets()


## Cell 3 — Teacher Embedding Extraction & Caching


In [ ]:
from torch.utils.data import TensorDataset

def encode_and_cache() -> Tuple[TensorDataset, torch.Tensor, torch.Tensor, np.ndarray, Dict]:
    """
    Extracts 384-dim MiniLM embeddings. Saves to disk; auto-resumes on restart.
    All training targets are in [0, 1] after Part II normalisation.
    """
    cache_path = os.path.join(WORKSPACE, 'data', 'embeddings_cache.pt')
    if os.path.exists(cache_path):
        print('Loading cached embeddings...')
        c = torch.load(cache_path, weights_only=False)
        return c['train_ds'], c['val_e1'], c['val_e2'], c['val_scores'], c['test_data']

    print('Encoding from scratch (one-time cost)...')
    teacher = SentenceTransformer('all-MiniLM-L6-v2')

    # STS-B train  ── cosine similarity mapped to [0,1]
    stsb_e1  = teacher.encode(train_df['sentence1'].tolist(), convert_to_tensor=True)
    stsb_e2  = teacher.encode(train_df['sentence2'].tolist(), convert_to_tensor=True)
    stsb_tgt = train_df['score'].values          # already in [0,1] after Part II fix
    stsb_tgt = torch.tensor(stsb_tgt, dtype=torch.float32)

    # AllNLI  ── cosine similarity already in [0,1] because cos∈[-1,1]→(cos+1)/2
    nli_anc  = teacher.encode(nli_df['anchor'].tolist(),   convert_to_tensor=True)
    nli_pos  = teacher.encode(nli_df['positive'].tolist(), convert_to_tensor=True)
    nli_neg  = teacher.encode(nli_df['negative'].tolist(), convert_to_tensor=True)
    pos_tgt  = (torch.cosine_similarity(nli_anc, nli_pos) + 1.0) / 2.0
    neg_tgt  = (torch.cosine_similarity(nli_anc, nli_neg) + 1.0) / 2.0

    all_e1  = torch.cat([stsb_e1,  nli_anc, nli_anc]).cpu()
    all_e2  = torch.cat([stsb_e2,  nli_pos, nli_neg]).cpu()
    all_tgt = torch.cat([stsb_tgt, pos_tgt, neg_tgt]).cpu()
    train_ds = TensorDataset(all_e1, all_e2, all_tgt)

    # Validation
    val_e1     = teacher.encode(val_df['sentence1'].tolist(), convert_to_tensor=True).cpu()
    val_e2     = teacher.encode(val_df['sentence2'].tolist(), convert_to_tensor=True).cpu()
    val_scores = val_df['score'].values          # [0,1]

    # Test sets
    test_data = {}
    for name, df, s1col, s2col in [
        ('BIOSSES', biosses_df, 'sentence1', 'sentence2'),
        ('STS-B',   stsb_test,  'sentence1', 'sentence2'),
    ]:
        e1 = teacher.encode(df[s1col].tolist(), convert_to_tensor=True).cpu()
        e2 = teacher.encode(df[s2col].tolist(), convert_to_tensor=True).cpu()
        sc = df['score'].values
        test_data[name] = {'e1': e1, 'e2': e2, 'scores': sc}

    torch.save(dict(train_ds=train_ds, val_e1=val_e1, val_e2=val_e2,
                    val_scores=val_scores, test_data=test_data), cache_path)
    print('Embeddings cached.')
    return train_ds, val_e1, val_e2, val_scores, test_data

train_ds, val_e1, val_e2, val_scores, test_data = encode_and_cache()
gc.collect(); torch.cuda.empty_cache()


## Cell 4 — Modular Architecture & Biophysical Surrogate


In [ ]:
class PearsonCorrelationLoss(nn.Module):
    """
    Optimises global rank-order alignment (Pearson r) between predicted
    DNA affinity and human semantic scores.
    Uses population variance (unbiased=False) for a consistent batch loss.
    """
    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        p = pred   - pred.mean()
        t = target - target.mean()
        cov     = (p * t).mean()
        pearson = cov / (p.std(unbiased=False) * t.std(unbiased=False) + 1e-8)
        return 1.0 - pearson


class ResidualMLPEncoder(nn.Module):
    """
    Maps 384-dim sentence embeddings → 128 × 4 one-hot DNA probabilities.
    Gumbel-Softmax relaxation allows end-to-end gradient flow through
    the discrete nucleotide sampling step.
    """
    def __init__(self, in_dim: int = 384, hidden: int = 512, seq_len: int = 128):
        super().__init__()
        self.seq_len = seq_len
        self.proj    = nn.Linear(in_dim, hidden)
        self.block1  = nn.Sequential(nn.Linear(hidden, hidden), nn.LayerNorm(hidden),
                                     nn.GELU(), nn.Dropout(0.1))
        self.block2  = nn.Sequential(nn.Linear(hidden, hidden), nn.LayerNorm(hidden),
                                     nn.GELU(), nn.Dropout(0.1))
        self.to_dna  = nn.Linear(hidden, seq_len * 4)

    def forward(self, x: torch.Tensor, tau: float = 1.0, hard: bool = False) -> torch.Tensor:
        h      = F.gelu(self.proj(x))
        h      = h + self.block1(h)
        h      = h + self.block2(h)
        logits = self.to_dna(h).view(-1, self.seq_len, 4)
        return F.gumbel_softmax(logits, tau=tau, hard=hard, dim=-1)


class BulletproofThermodynamicSurrogate(nn.Module):
    """
    Differentiable DNA hybridisation energy surrogate.

    Physical model:
      ΔG  = ΔH − T·ΔS  (SantaLucia 1998 nearest-neighbour parameters)
      T   = 348.15 K   (75°C high-stringency hybridisation)

    Biophysical constraints:
      - Watson-Crick complementarity enforced via reversed-complement tensor.
      - Sequence-dependent wobble/mismatch matrix (G-T = 0.5 kcal/mol;
        purine–purine clashes = 2.0 kcal/mol).
      - Hairpin/secondary-structure penalty via self-complementarity dot product.

    FIX (critical): mismatch penalty now correctly compares dna1 against
    dna2_c (the Watson-Crick complement of dna2) so that A-T and G-C pairs
    receive zero penalty and mismatches are penalised proportionally.

    FIX (performance): nearest-neighbour ΔG loop fully vectorised using
    batch einsum over successive dinucleotide pairs, eliminating 127
    sequential Python iterations.
    """
    # Nucleotide ordering: A=0  C=1  G=2  T=3
    _dH = [[-7.9,-8.4,-7.8,-7.2],
           [-8.5,-8.0,-10.6,-7.8],
           [-8.2,-9.8,-8.0,-8.4],
           [-7.2,-8.2,-8.5,-7.9]]

    _dS = [[-22.2,-22.4,-21.0,-20.4],
           [-22.7,-19.9,-27.2,-21.0],
           [-22.2,-24.4,-19.9,-22.4],
           [-21.3,-22.2,-22.7,-22.2]]

    # Penalty for each dna1-base vs. dna2-complement-base pair.
    # Watson-Crick pairs (A-T, T-A, C-G, G-C) → 0.0
    # G-T wobble → 0.5;  purine-purine clash → 2.0
    _MM = [[0.0, 1.5, 2.0, 2.0],   # A vs A,C,G,T complement
           [1.5, 0.0, 2.0, 0.5],   # C vs ...
           [2.0, 2.0, 0.0, 0.5],   # G vs ...
           [2.0, 0.5, 0.5, 0.0]]   # T vs ...

    def __init__(self, seq_len: int = 128, temperature: float = 348.15):
        super().__init__()
        self.seq_len = seq_len
        self.T = temperature
        self.register_buffer('dH', torch.tensor(self._dH))
        self.register_buffer('dS', torch.tensor(self._dS))
        self.register_buffer('MM', torch.tensor(self._MM))

    # ── helpers ──────────────────────────────────────────────────────
    @staticmethod
    def _complement(dna: torch.Tensor) -> torch.Tensor:
        """Watson-Crick complement: swap A↔T (0↔3) and C↔G (1↔2)."""
        return dna[:, :, [3, 2, 1, 0]]

    def _hairpin_penalty(self, dna: torch.Tensor) -> torch.Tensor:
        """Detects self-complementarity (hairpin potential) via batch matmul."""
        rc = torch.flip(self._complement(dna), dims=[1])          # reverse complement
        return torch.bmm(dna.view(-1, self.seq_len, 4),
                         rc.view(-1, self.seq_len, 4).transpose(1, 2)).mean(dim=(1, 2))

    # ── forward ──────────────────────────────────────────────────────
    def forward(self, dna1: torch.Tensor, dna2: torch.Tensor) -> torch.Tensor:
        dna2_c = self._complement(dna2)                           # (B, L, 4)

        # ── Watson-Crick match mask per position ─────────────────────
        match = (dna1 * dna2_c).sum(-1)                          # (B, L)

        # ── FIX: vectorised ΔG (no Python loop) ──────────────────────
        # Successive-pair outer products: shape (B, L-1, 4, 4)
        d1_l  = dna1[:,  :-1, :]                                  # (B, L-1, 4)
        d1_r  = dna1[:,  1:,  :]
        d2_l  = dna2_c[:, :-1, :]
        d2_r  = dna2_c[:, 1:,  :]

        outer1 = torch.einsum('bni,bnj->bnij', d1_l, d1_r)       # dimer from strand 1
        outer2 = torch.einsum('bni,bnj->bnij', d2_l, d2_r)       # dimer from strand 2

        dG_mat = (self.dH - self.T * self.dS / 1000.0)           # (4,4)
        step_e = torch.einsum('bnij,ij,bnij->bn', outer1, dG_mat, outer2)

        pair_mask = match[:, :-1] * match[:, 1:]                 # both positions matched
        nn_energy = (step_e * pair_mask).sum(dim=1)              # (B,)

        # ── FIX: mismatch penalty uses dna2_c not dna2 ───────────────
        mismatch = torch.einsum('bni,ij,bnj->b', dna1, self.MM, dna2_c)

        # ── Hairpin penalty ───────────────────────────────────────────
        hairpin  = self._hairpin_penalty(dna1) + self._hairpin_penalty(dna2)

        total = nn_energy + mismatch + 2.0 * hairpin

        # Negate: more negative total energy → higher binding affinity score
        return -total


print('Architecture defined.')
print(f'  ResidualMLPEncoder  : ~{sum(p.numel() for p in ResidualMLPEncoder().parameters()):,} params')
print(f'  ThermodynamicSurrogate: 0 learnable params (fixed physical constants)')


## Cell 5 — Training Pipeline (Cosine-Annealing LR + Early Stopping)


In [ ]:
from torch.utils.data import DataLoader

encoder   = ResidualMLPEncoder().to(device)
predictor = BulletproofThermodynamicSurrogate().to(device)
criterion = PearsonCorrelationLoss()
optimizer = torch.optim.AdamW(encoder.parameters(), lr=5e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True, drop_last=True)
ckpt_path    = os.path.join(WORKSPACE, 'models', 'checkpoint.pth')

best_rho, patience_ctr, PATIENCE = -1.0, 0, 5

print('Starting training...')
for epoch in range(1, 26):
    encoder.train()
    tau   = max(0.1, 1.0 - epoch * 0.05)   # anneal Gumbel temperature 1.0→0.1
    total = 0.0

    for b_e1, b_e2, b_tgt in train_loader:
        optimizer.zero_grad()
        d1  = encoder(b_e1.to(device), tau=tau)
        d2  = encoder(b_e2.to(device), tau=tau)
        aff = predictor(d1, d2)
        loss = criterion(aff, b_tgt.to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), 1.0)
        optimizer.step()
        total += loss.item()

    scheduler.step()

    # ── Validation (hard discrete sequences) ─────────────────────────
    encoder.eval()
    with torch.no_grad():
        v_d1  = encoder(val_e1.to(device), hard=True)
        v_d2  = encoder(val_e2.to(device), hard=True)
        v_aff = predictor(v_d1, v_d2).cpu().numpy()

    val_rho, _ = stats.spearmanr(val_scores, v_aff)

    if val_rho > best_rho:
        best_rho, patience_ctr = val_rho, 0
        torch.save({'encoder': encoder.state_dict(), 'best_rho': best_rho}, ckpt_path)
        tag = 'SAVED'
    else:
        patience_ctr += 1
        tag = f'({patience_ctr}/{PATIENCE})'

    print(f'Ep {epoch:02d} | loss {total/len(train_loader):.4f} | val rho {val_rho:.4f} | {tag}')
    if patience_ctr >= PATIENCE:
        print('Early stop.')
        break

ck = torch.load(ckpt_path, weights_only=False)
encoder.load_state_dict(ck['encoder'])
print(f'Best model loaded  (val rho = {ck["best_rho"]:.4f})')


## Cell 6 — Zero-Shot Generalisation (BIOSSES & STS-B)


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
encoder.eval()

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=300)

for ax, (name, color) in zip(axes, [('STS-B','#2ca02c'), ('BIOSSES','#d62728')]):
    td = test_data[name]
    with torch.no_grad():
        d1  = encoder(td['e1'].to(device), hard=True)
        d2  = encoder(td['e2'].to(device), hard=True)
        aff = predictor(d1, d2).cpu().numpy()

    sc = td['scores']

    # FIX: compute real p-value instead of hardcoding
    rho, pval = stats.spearmanr(sc, aff)
    p_str = f'{pval:.2e}'

    ax.scatter(aff, sc, alpha=0.5, s=20, c=color, edgecolors='k', linewidths=0.3)
    m, b = np.polyfit(aff, sc, 1)
    ax.plot(aff, m*aff + b, 'k--', lw=2, label=f'Trend  rho={rho:.3f}  p={p_str}')
    ax.set_title(f'Zero-Shot: {name} (n={len(sc)})', fontweight='bold')
    ax.set_xlabel('Predicted DNA Hybridisation Affinity  (-DeltaG)')
    ax.set_ylabel('Human Semantic Score  [0–1]')
    ax.legend()

plt.tight_layout()
path = os.path.join(WORKSPACE, 'figures', 'zero_shot.png')
plt.savefig(path)
plt.show()
print(f'Saved: {path}')


## Cell 7 — Baseline Comparison (6 Methods)


In [ ]:
td     = test_data['STS-B']
scores = td['scores']
e1_np  = td['e1'].numpy()
e2_np  = td['e2'].numpy()

results = {}

# 1. Float32 teacher (upper bound)
cos = torch.cosine_similarity(td['e1'], td['e2']).numpy()
results['Float32 Teacher (UB)'] = stats.spearmanr(scores, cos)[0]

# 2. Int8 scalar quantisation
def quant_int8(x):
    lo, hi = x.min(), x.max()
    s = (hi - lo) / 255.0 + 1e-9
    return np.round((x - lo) / s) * s + lo

i8sim = np.einsum('bi,bi->b', quant_int8(e1_np), quant_int8(e2_np)) / (
        np.linalg.norm(quant_int8(e1_np), axis=1) *
        np.linalg.norm(quant_int8(e2_np), axis=1) + 1e-9)
results['Int8 Quantisation'] = stats.spearmanr(scores, i8sim)[0]

# 3. Binary quantisation (sign binarisation)
b1 = (e1_np > 0).astype(np.float32)
b2 = (e2_np > 0).astype(np.float32)
results['Binary Quantisation'] = stats.spearmanr(
    scores, 1.0 - np.mean(b1 != b2, axis=1))[0]

# 4. LSH (256-bit Gaussian random projections)
rp = GaussianRandomProjection(n_components=256, random_state=42)
rp.fit(np.vstack([e1_np, e2_np]))
l1 = (rp.transform(e1_np) > 0).astype(np.float32)
l2 = (rp.transform(e2_np) > 0).astype(np.float32)
results['LSH Hashing (256-bit)'] = stats.spearmanr(
    scores, 1.0 - np.mean(l1 != l2, axis=1))[0]

# 5. Random DNA (lower bound)
encoder.eval()
with torch.no_grad():
    r1  = F.one_hot(torch.randint(0,4,(td['e1'].size(0),128),device=device),4).float()
    r2  = F.one_hot(torch.randint(0,4,(td['e2'].size(0),128),device=device),4).float()
    results['Random DNA (LB)'] = stats.spearmanr(
        scores, predictor(r1, r2).cpu().numpy())[0]

# 6. ChemiSearch (ours)
with torch.no_grad():
    d1 = encoder(td['e1'].to(device), hard=True)
    d2 = encoder(td['e2'].to(device), hard=True)
    results['ChemiSearch (Ours)'] = stats.spearmanr(
        scores, predictor(d1, d2).cpu().numpy())[0]

# ── Print table ───────────────────────────────────────────────────────
order = ['Random DNA (LB)','Binary Quantisation','LSH Hashing (256-bit)',
         'Int8 Quantisation','Float32 Teacher (UB)','ChemiSearch (Ours)']
print(f'{"Method":<28}  Spearman rho')
print('-' * 42)
for k in order:
    print(f'{k:<28}  {results[k]:+.4f}')

# ── Bar chart ─────────────────────────────────────────────────────────
rhos   = [results[k] for k in order]
colors = ['#7f7f7f','#1f77b4','#1f77b4','#1f77b4','#2ca02c','#d62728']

fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
ax.barh(order, rhos, color=colors, edgecolor='black', alpha=0.9)
ax.set_xlabel('Semantic Preservation  (Spearman rho)', fontweight='bold')
ax.set_title('Information Preserved: Biophysical DNA vs Digital Compression Methods',
             fontweight='bold', fontsize=13)
ax.grid(axis='x', linestyle='--', alpha=0.6)
for i, v in enumerate(rhos):
    ax.text(max(v, 0.0) + 0.01, i, f'{v:.3f}', va='center', fontweight='bold')
plt.xlim(-0.1, 1.0)
plt.tight_layout()
path = os.path.join(WORKSPACE, 'figures', 'baselines.png')
plt.savefig(path)
plt.show()
print(f'Saved: {path}')


## Cell 8 — Biological Validity & Robustness Stress Test


In [ ]:
encoder.eval()
with torch.no_grad():
    d1 = encoder(test_data['STS-B']['e1'].to(device), hard=True)
    d2 = encoder(test_data['STS-B']['e2'].to(device), hard=True)

# ── GC content ────────────────────────────────────────────────────────
# Nucleotide order: A=0  C=1  G=2  T=3
gc_pct = (d1[:,:,1] + d1[:,:,2]).sum(1).cpu().numpy() / 128.0 * 100.0

# ── Mutation robustness ────────────────────────────────────────────────
# Use full test set with bootstrapped CI for statistical reliability
sc_full    = test_data['STS-B']['scores']
N          = len(sc_full)
error_rates = [0.0, 0.02, 0.05, 0.10, 0.15, 0.20]
rho_mean, rho_lo, rho_hi = [], [], []

BOOTS = 200
for er in error_rates:
    boot_rhos = []
    for _ in range(BOOTS):
        idx = np.random.choice(N, N, replace=True)
        nm1 = torch.rand(N, 128, device=device) < er
        nm2 = torch.rand(N, 128, device=device) < er
        rb1 = F.one_hot(torch.randint(0,4,(N,128),device=device),4).float()
        rb2 = F.one_hot(torch.randint(0,4,(N,128),device=device),4).float()
        md1 = torch.where(nm1.unsqueeze(-1), rb1, d1)
        md2 = torch.where(nm2.unsqueeze(-1), rb2, d2)
        with torch.no_grad():
            aff = predictor(md1, md2).cpu().numpy()
        r, _ = stats.spearmanr(sc_full[idx], aff[idx])
        boot_rhos.append(r)
    rho_mean.append(np.mean(boot_rhos))
    rho_lo.append(np.percentile(boot_rhos, 2.5))
    rho_hi.append(np.percentile(boot_rhos, 97.5))

# ── Figures ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

# A: GC content
axes[0].hist(gc_pct, bins=25, color='#9467bd', edgecolor='k', alpha=0.75, density=True)
axes[0].axvline(50, color='r', ls='--', label='Target 50%')
axes[0].axvspan(40, 60, color='green', alpha=0.1, label='Optimal range (40-60%)')
axes[0].set_title('A. GC Content Distribution', fontweight='bold')
axes[0].set_xlabel('GC Content (%)')
axes[0].set_ylabel('Density')
axes[0].legend()

# B: Robustness with 95% CI ribbons
er_pct = [e * 100 for e in error_rates]
axes[1].plot(er_pct, rho_mean, 'o-', lw=2, color='#e377c2', label='Mean rho')
axes[1].fill_between(er_pct, rho_lo, rho_hi, alpha=0.25, color='#e377c2',
                     label='95% Bootstrap CI')
axes[1].set_title('B. Robustness to Synthesis/Sequencing Errors  (bootstrapped)', fontweight='bold')
axes[1].set_xlabel('Mutation Rate (%)')
axes[1].set_ylabel('Spearman rho')
axes[1].legend()
axes[1].grid(True, ls='--', alpha=0.5)

plt.tight_layout()
path = os.path.join(WORKSPACE, 'figures', 'bio_validity.png')
plt.savefig(path)
plt.show()
print(f'Saved: {path}')


## Cell 9 — Real-World Molecular Case Study


In [ ]:
COMP = {'A':'T', 'T':'A', 'C':'G', 'G':'C'}

def to_dna(onehot: torch.Tensor) -> str:
    """Decode one-hot tensor to nucleotide string (5' to 3')."""
    return ''.join('ACGT'[i] for i in onehot.argmax(-1).cpu().tolist())

def reverse_complement(seq: str) -> str:
    """True Watson-Crick reverse complement."""
    return ''.join(COMP[b] for b in reversed(seq))

texts = [
    "A patient diagnosed with severe hypertension.",
    "The individual is suffering from very high blood pressure.",
    "The cat is sleeping peacefully on the sofa."
]

teacher = SentenceTransformer('all-MiniLM-L6-v2')
embs    = teacher.encode(texts, convert_to_tensor=True).to(device)

encoder.eval()
with torch.no_grad():
    dna_oh = encoder(embs, hard=True)

seqs = [to_dna(dna_oh[i]) for i in range(3)]

with torch.no_grad():
    aff_12 = predictor(dna_oh[0:1], dna_oh[1:2]).item()
    aff_13 = predictor(dna_oh[0:1], dna_oh[2:3]).item()

print('-' * 80)
print('[QUERY]')
print(f'  Text : {texts[0]}')
print(f"  DNA  : 5'-{seqs[0]}-3'")
print()
print('[TARGET 1  —  Semantic Near-Duplicate]')
print(f'  Text : {texts[1]}')
# FIX: display true reverse complement, not just reversed string
rc1 = reverse_complement(seqs[1])
print(f"  DNA  : 3'-{rc1}-5'  (true Watson-Crick reverse complement)")
print(f'  Predicted binding affinity (-DeltaG) : {aff_12:.2f}')
print()
print('[TARGET 2  —  Semantic Mismatch]')
print(f'  Text : {texts[2]}')
rc2 = reverse_complement(seqs[2])
print(f"  DNA  : 3'-{rc2}-5'  (true Watson-Crick reverse complement)")
print(f'  Predicted binding affinity (-DeltaG) : {aff_13:.2f}')
print()
print('Interpretation:')
print(f'  Higher value = more stable hybridisation at 75 C.')
print(f'  Semantic match vs mismatch delta = {aff_12 - aff_13:.2f}')
print('-' * 80)


## Cell 10 — Final Export


## Cell 11 — Ablation Study (Figure 5)
Systematically disables each architectural component to demonstrate that every part
of the model contributes meaningful performance. This is a mandatory requirement
for Q1 journal submission.


In [ ]:
# =============================================================================
# Cell 11: Ablation Study
# =============================================================================
# Five configurations evaluated on the STS-B test set (Spearman rho).
# Each variant disables exactly one architectural component.
# The predictor (thermodynamic surrogate) is the same across all variants.
# =============================================================================

import copy

td     = test_data['STS-B']
scores = td['scores']
SEQ_LEN = 128
IN_DIM  = 384
HIDDEN  = 512

# ---------------------------------------------------------------------------
# Helper: train and evaluate a given encoder configuration
# ---------------------------------------------------------------------------

def train_and_evaluate(
    enc_model: torch.nn.Module,
    tag: str,
    epochs: int = 8,
    patience: int = 3,
) -> float:
    """
    Trains enc_model for a fixed number of epochs using the same
    training data, loss, and optimizer settings as the main pipeline.
    Returns the best validation Spearman rho achieved.
    Note: fewer epochs than the main run — sufficient to assess relative
    capacity, not to reach convergence.
    """
    from torch.utils.data import DataLoader
    enc = enc_model.to(device)
    opt = torch.optim.AdamW(enc.parameters(), lr=5e-4, weight_decay=1e-3)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = PearsonCorrelationLoss()
    loader = DataLoader(train_ds, batch_size=512, shuffle=True, drop_last=True)

    best_rho = -1.0
    pat_ctr  = 0

    for epoch in range(1, epochs + 1):
        enc.train()
        tau = max(0.1, 1.0 - epoch * 0.05)
        for b_e1, b_e2, b_tgt in loader:
            opt.zero_grad()
            d1  = enc(b_e1.to(device), tau=tau)
            d2  = enc(b_e2.to(device), tau=tau)
            aff = predictor(d1, d2)
            loss = crit(aff, b_tgt.to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(enc.parameters(), 1.0)
            opt.step()
        sch.step()

        enc.eval()
        with torch.no_grad():
            v_d1  = enc(val_e1.to(device), hard=True)
            v_d2  = enc(val_e2.to(device), hard=True)
            v_aff = predictor(v_d1, v_d2).cpu().numpy()
        rho, _ = stats.spearmanr(val_scores, v_aff)

        if rho > best_rho:
            best_rho = rho
            pat_ctr  = 0
        else:
            pat_ctr += 1
        if pat_ctr >= patience:
            break

    # Final test evaluation using best checkpoint approximation (last best)
    enc.eval()
    with torch.no_grad():
        t_d1  = enc(td['e1'].to(device), hard=True)
        t_d2  = enc(td['e2'].to(device), hard=True)
        t_aff = predictor(t_d1, t_d2).cpu().numpy()
    test_rho, _ = stats.spearmanr(scores, t_aff)

    print(f"  {tag:<40}  test rho = {test_rho:.4f}")
    return test_rho


# ---------------------------------------------------------------------------
# Variant A: Full Model (the fixed model already trained in Cell 5)
# ---------------------------------------------------------------------------
print("Ablation Study — STS-B Spearman rho")
print("-" * 55)

encoder.eval()
with torch.no_grad():
    full_d1  = encoder(td['e1'].to(device), hard=True)
    full_d2  = encoder(td['e2'].to(device), hard=True)
    full_aff = predictor(full_d1, full_d2).cpu().numpy()
rho_full, _ = stats.spearmanr(scores, full_aff)
print(f"  {'Full Model (Ours)':<40}  test rho = {rho_full:.4f}")

# ---------------------------------------------------------------------------
# Variant B: No Hairpin Penalty
# Subclass the surrogate and zero-out the hairpin term.
# ---------------------------------------------------------------------------
class NoHairpinSurrogate(BulletproofThermodynamicSurrogate):
    def forward(self, dna1, dna2):
        dna2_c   = self._complement(dna2)
        match    = (dna1 * dna2_c).sum(-1)
        d1_l, d1_r = dna1[:, :-1], dna1[:, 1:]
        d2_l, d2_r = dna2_c[:, :-1], dna2_c[:, 1:]
        o1 = torch.einsum('bni,bnj->bnij', d1_l, d1_r)
        o2 = torch.einsum('bni,bnj->bnij', d2_l, d2_r)
        dG = self.dH - self.T * self.dS / 1000.0
        step_e = torch.einsum('bnij,ij,bnij->bn', o1, dG, o2)
        mask   = match[:, :-1] * match[:, 1:]
        nn_e   = (step_e * mask).sum(1)
        mismatch = torch.einsum('bni,ij,bnj->b', dna1, self.MM, dna2_c)
        return -(nn_e + mismatch)   # hairpin term omitted

no_hp_pred = NoHairpinSurrogate().to(device)
enc_no_hp  = ResidualMLPEncoder().to(device)
rho_no_hp  = train_and_evaluate(enc_no_hp, "No Hairpin Penalty")

# ---------------------------------------------------------------------------
# Variant C: No Mismatch Penalty
# ---------------------------------------------------------------------------
class NoMismatchSurrogate(BulletproofThermodynamicSurrogate):
    def forward(self, dna1, dna2):
        dna2_c = self._complement(dna2)
        match  = (dna1 * dna2_c).sum(-1)
        d1_l, d1_r = dna1[:, :-1], dna1[:, 1:]
        d2_l, d2_r = dna2_c[:, :-1], dna2_c[:, 1:]
        o1 = torch.einsum('bni,bnj->bnij', d1_l, d1_r)
        o2 = torch.einsum('bni,bnj->bnij', d2_l, d2_r)
        dG = self.dH - self.T * self.dS / 1000.0
        step_e = torch.einsum('bnij,ij,bnij->bn', o1, dG, o2)
        mask   = match[:, :-1] * match[:, 1:]
        nn_e   = (step_e * mask).sum(1)
        hairpin = self._hairpin_penalty(dna1) + self._hairpin_penalty(dna2)
        return -(nn_e + 2.0 * hairpin)   # mismatch term omitted

no_mm_pred = NoMismatchSurrogate().to(device)
enc_no_mm  = ResidualMLPEncoder().to(device)
rho_no_mm  = train_and_evaluate(enc_no_mm, "No Mismatch Penalty")

# ---------------------------------------------------------------------------
# Variant D: No Residual Connections (shallow linear encoder)
# ---------------------------------------------------------------------------
class LinearEncoder(torch.nn.Module):
    """Direct linear projection with no residual connections or normalisation."""
    def __init__(self, in_dim=384, hidden=512, seq_len=128):
        super().__init__()
        self.seq_len = seq_len
        self.fc1 = torch.nn.Linear(in_dim, hidden)
        self.fc2 = torch.nn.Linear(hidden, seq_len * 4)

    def forward(self, x, tau=1.0, hard=False):
        h      = torch.relu(self.fc1(x))
        logits = self.fc2(h).view(-1, self.seq_len, 4)
        return torch.nn.functional.gumbel_softmax(logits, tau=tau, hard=hard, dim=-1)

enc_linear = LinearEncoder().to(device)
rho_linear = train_and_evaluate(enc_linear, "No Residual Blocks (Linear)")

# ---------------------------------------------------------------------------
# Variant E: Random DNA lower bound (no learning)
# ---------------------------------------------------------------------------
with torch.no_grad():
    r1 = torch.nn.functional.one_hot(
        torch.randint(0, 4, (td['e1'].size(0), SEQ_LEN), device=device), 4).float()
    r2 = torch.nn.functional.one_hot(
        torch.randint(0, 4, (td['e2'].size(0), SEQ_LEN), device=device), 4).float()
    rand_aff = predictor(r1, r2).cpu().numpy()
rho_rand, _ = stats.spearmanr(scores, rand_aff)
print(f"  {'Random DNA (Lower Bound)':<40}  test rho = {rho_rand:.4f}")

# ---------------------------------------------------------------------------
# Figure 5: Ablation bar chart
# ---------------------------------------------------------------------------
variants = [
    ("Random DNA (LB)",        rho_rand),
    ("No Residual Blocks",     rho_linear),
    ("No Mismatch Penalty",    rho_no_mm),
    ("No Hairpin Penalty",     rho_no_hp),
    ("Full Model (Ours)",      rho_full),
]

names = [v[0] for v in variants]
rhos  = [v[1] for v in variants]
colors = ['#7f7f7f', '#aec7e8', '#ffbb78', '#98df8a', '#d62728']

fig, ax = plt.subplots(figsize=(9, 5), dpi=300)
bars = ax.barh(names, rhos, color=colors, edgecolor='black', alpha=0.9)
ax.set_xlabel('STS-B Spearman rho (Test)', fontweight='bold', fontsize=12)
ax.set_title('Figure 5: Ablation Study — Contribution of Each Component',
             fontweight='bold', fontsize=13)
ax.grid(axis='x', linestyle='--', alpha=0.6)
ax.axvline(rho_rand, color='gray', linestyle=':', linewidth=1.5, label='Random baseline')

for i, v in enumerate(rhos):
    ax.text(max(v, 0.0) + 0.005, i, f'{v:.4f}', va='center', fontweight='bold', fontsize=11)

ax.set_xlim(-0.05, max(rhos) + 0.12)
plt.tight_layout()
path = os.path.join(WORKSPACE, 'figures', 'ablation_study.png')
plt.savefig(path)
plt.show()
print(f"Ablation figure saved to {path}")


In [ ]:
import shutil
from IPython.display import FileLink

export = os.path.join(WORKSPACE, 'export')
os.makedirs(export, exist_ok=True)
shutil.copytree(os.path.join(WORKSPACE,'figures'), os.path.join(export,'figures'),
                dirs_exist_ok=True)
shutil.copytree(os.path.join(WORKSPACE,'models'),  os.path.join(export,'models'),
                dirs_exist_ok=True)

zip_base = os.path.join(WORKSPACE, 'DNA_Semantic_Search_Release')
shutil.make_archive(zip_base, 'zip', export)

print('All artifacts packaged.')
display(FileLink('dna_search_workspace/DNA_Semantic_Search_Release.zip'))
